# 意味変化の検出手法を評価データで比べる (SemEval-2020 Task 1)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/18_lexical_semantics/semantic_change_eval.ipynb)

NLP クックブック 18-15 ／ 語の意味と意味の変化

意味変化の度合いを測る手法が、人手で付けた「変化の大きさ」とどれくらい合うかを確かめ、手法どうしを比べたい。

In [1]:
!pip install -q gensim
!wget -q https://www2.ims.uni-stuttgart.de/data/sem-eval-ulscd/semeval2020_ulscd_eng.zip && unzip -q -o semeval2020_ulscd_eng.zip

In [2]:
import gzip
import numpy as np
from gensim.models import Word2Vec
from scipy.stats import spearmanr

base = "semeval2020_ulscd_eng/"
gold = dict(line.split("\t") for line in open(base + "truth/graded.txt").read().strip().split("\n"))
gold = {w: float(v) for w, v in gold.items()}                 # 対象語 37 語と、人手の注釈から求めた変化の度合い (0〜1)
corpora = {}
for name, path in [("1810-1860", "corpus1/lemma/ccoha1.txt.gz"), ("1960-2010", "corpus2/lemma/ccoha2.txt.gz")]:
    with gzip.open(base + path, "rt", encoding="utf-8") as f:
        corpora[name] = [line.split() for line in f]          # 見出し語化済み。対象語には attack_nn のように品詞が付いている
    print(name, len(corpora[name]), "文", sum(map(len, corpora[name])), "語")
print(sorted(gold.items(), key=lambda x: -x[1])[:6], "...")

1810-1860 253644 文 6559657 語
1960-2010 353692 文 6769814 語
[('plane_nn', 0.8823475834), ('tip_vb', 0.6788987615), ('prop_nn', 0.624759971), ('graft_nn', 0.5539755971), ('record_nn', 0.4273504707), ('ball_nn', 0.4093665525)] ...


In [3]:
c1, c2 = corpora.values()
models = [Word2Vec(c, vector_size=100, window=10, min_count=10, sg=1, negative=5, epochs=5, workers=4, seed=0).wv for c in (c1, c2)]

# 手法 1: 直交プロクラステスで位置合わせしてコサイン距離
vocab = [w for w in models[0].index_to_key if w in models[1].key_to_index]
A = np.stack([models[0][w] for w in vocab]); A = (A - A.mean(0)); A /= np.linalg.norm(A, axis=1, keepdims=True)
B = np.stack([models[1][w] for w in vocab]); B = (B - B.mean(0)); B /= np.linalg.norm(B, axis=1, keepdims=True)
U, _, Vt = np.linalg.svd(A.T @ B)
index = {w: i for i, w in enumerate(vocab)}
procrustes = {w: 1 - float((A[index[w]] @ U @ Vt) @ B[index[w]]) for w in gold}

# 手法 2: 近傍語の入れ替わり (位置合わせなし)
def neighbor_change(word, k=50):
    shared = set(vocab)
    near = [[w for w, _ in m.most_similar(word, topn=k * 3) if w in shared][:k] for m in models]
    ref = sorted(set(near[0]) | set(near[1]))
    a, b = (np.array([m.similarity(word, r) for r in ref]) for m in models)
    return 1 - float(a @ b / np.linalg.norm(a) / np.linalg.norm(b))
neighbor = {w: neighbor_change(w) for w in gold}

# ベースライン: 相対頻度の変化
count = [{w: m.get_vecattr(w, "count") / sum(map(len, c)) for w in gold} for m, c in zip(models, (c1, c2))]
frequency = {w: abs(np.log(count[1][w] / count[0][w])) for w in gold}

In [4]:
words = sorted(gold)
for name, scores in [("頻度の変化 (ベースライン)", frequency), ("近傍語の入れ替わり", neighbor), ("位置合わせ + コサイン距離", procrustes)]:
    r = spearmanr([gold[w] for w in words], [scores[w] for w in words])
    print(f"{name}: Spearman = {r.statistic:.3f} (p = {r.pvalue:.3f})")

print("正解で変化が大きい語:", sorted(gold, key=gold.get, reverse=True)[:8])
print("手法が変化が大きいとした語:", sorted(procrustes, key=procrustes.get, reverse=True)[:8])
for w in ["plane_nn", "tip_vb", "chairman_nn"]:
    print(w, round(gold[w], 2), "|", [x for x, _ in models[0].most_similar(w, topn=5)], "→", [x for x, _ in models[1].most_similar(w, topn=5)])

頻度の変化 (ベースライン): Spearman = -0.279 (p = 0.095)
近傍語の入れ替わり: Spearman = 0.232 (p = 0.168)
位置合わせ + コサイン距離: Spearman = 0.301 (p = 0.071)
正解で変化が大きい語: ['plane_nn', 'tip_vb', 'prop_nn', 'graft_nn', 'record_nn', 'ball_nn', 'stab_nn', 'twist_nn']
手法が変化が大きいとした語: ['plane_nn', 'record_nn', 'graft_nn', 'prop_nn', 'savage_nn', 'multitude_nn', 'player_nn', 'twist_nn']
plane_nn 0.88 | ['projection', 'horizontal', 'prism', 'vertical', 'intersection'] → ['runway', 'pilot', 'helicopter', 'airplane', 'fly']
tip_vb 0.68 | ['decanter', 'knob', 'cracked', 'squint', 'trim'] → ['transom', 'hoist', 'dribble', 'dud', 'untie']
chairman_nn 0.0 | ['committee', 'nominate', 'moderator', 'teller', 'selectmen'] → ['vice', 'treasurer', 'undersecretary', 'blumenthal', 'stennis']


- SemEval-2020 Task 1 は、2 つの時代のコーパスと、対象語ごとの変化の度合い（人手の注釈から計算）を提供する。手法の出したスコアとの順位相関 (Spearman) で評価する。英語のほか、ドイツ語、ラテン語、スウェーデン語がある。
- 対象語が 37 語しかないので、相関係数の信頼区間は広い。0.1 程度の差は、乱数シードを変えるだけで入れ替わる。複数のシードと複数の言語で確かめる。
- 「頻度が変わった語は意味も変わった」という単純なベースラインと必ず比べる。これを大きく超えられない手法は、意味ではなく頻度を見ているだけかもしれない。
- このタスクでは、単語ベクトルを位置合わせする古典的な方法が、文脈化ベクトルを使う方法と同等以上の成績を出したことが知られている。新しい手法が常に良いとは限らない。
- 日本語には、日本語歴史コーパスなどを使った評価データ (JaSemChange) がある。コーパスの利用申請が必要。